In [49]:
import torch 
import torch.nn as nn 
import torch.optim as optim

In [50]:
import torchvision
from torch.utils.data import DataLoader

In [51]:
import torchvision.transforms as transforms

transform=transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

In [52]:
from torchvision import datasets

dataset=datasets.ImageFolder("data/PetImages",transform=transform)


In [53]:
from torch.utils.data import random_split



train_size=int(0.8*len(dataset))

test_size=int(len(dataset)-train_size)

In [54]:
train_size

19998

In [55]:
test_size

5000

In [56]:
trainset,testset=random_split(
    dataset,[train_size,test_size]
)

In [57]:
trainset

In [58]:
testset

In [59]:
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)
testloader=DataLoader(testset,batch_size=32)

### Build the CNN

In [60]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(16*16*128,256),
            nn.ReLU(),

            nn.Linear(256,2)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x


In [61]:
model=CNN()

In [62]:
criteria=nn.CrossEntropyLoss()

In [63]:
optimizer=optim.Adam(model.parameters())

### Training the CNN

In [64]:
epochs=10

model.train()

for epoch in range(epochs):

    epoch_training_loss=0.0

    for images,labels in trainloader:

        optimizer.zero_grad()

        output=model.forward(images)

        loss=criteria(output,labels)

        loss.backward()

        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} => training loss is {epoch_training_loss/len(trainloader)}")

c:\Users\HP\anaconda3\envs\deep_learning\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


1/10 => training loss is 0.6054721505641937
2/10 => training loss is 0.4830505982875824
3/10 => training loss is 0.4040835147023201
4/10 => training loss is 0.32958913739919665
5/10 => training loss is 0.23325186386108399
6/10 => training loss is 0.12696155676692725
7/10 => training loss is 0.0627466803457588
8/10 => training loss is 0.037486522436980155
9/10 => training loss is 0.03187403919682838
10/10 => training loss is 0.03896665114262141


In [65]:
# Evaluation

model.eval()

correct=0
total=0
with torch.no_grad():
    for images, labels in testloader:

        outputs=model.forward(images)

        _,predicted=torch.max(outputs,1)

        correct+=(predicted==labels).sum().item()

        total+=labels.size(0)

        accuracy=correct/total*100

print(accuracy)

81.06
